# 01 — 데이터 수집

**REFRESH_DATA = True** 로 설정해야 API를 재수집합니다.  
**REFRESH_DATA = False** (기본값) 이면 `data/` 폴더의 parquet을 그대로 로드합니다.

> 한 번 수집한 데이터는 다시 긁어오지 말 것 — 종목 구성·API 응답이 달라져 결과가 바뀜.

## 0. 설치

In [1]:
# !pip install finance-datareader yfinance statsmodels tqdm -q
# !apt-get install -y fonts-nanum -q

## 1. 파라미터

In [2]:
import pyarrow  # pandas Period 타입 충돌 방지: pandas보다 먼저 import
from dotenv import load_dotenv
import os
load_dotenv()
DART_API_KEY = os.environ.get("DART_API_KEY")  # .env에서 로드

import warnings; warnings.filterwarnings('ignore')
import platform, time, json as _json
from pathlib import Path

import pandas as pd
import numpy as np
import FinanceDataReader as fdr
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# ── 파라미터 ──────────────────────────────────────────────────────────────────
DATE_START       = '2015-01-01'
DATE_END         = '2026-03-31'
BETA_MIN_PERIODS = 36    # 최소 36개월치 있어야 수집 대상으로 포함
N_WORKERS        = 20    # yfinance 병렬 스레드 수

# ── 데이터 관리 ──────────────────────────────────────────────────────────────
# REFRESH_DATA = True  → API 재수집 후 parquet 덮어쓰기
# REFRESH_DATA = False → 저장된 parquet 로드만 (기본값)
REFRESH_DATA = False
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

print(f'FDR {fdr.__version__} | yfinance {yf.__version__}')
print(f'분석 기간: {DATE_START} ~ {DATE_END}')
print(f'REFRESH_DATA = {REFRESH_DATA}')
print(f'DATA_DIR = {DATA_DIR.resolve()}')


FDR 0.9.110 | yfinance 1.2.0
분석 기간: 2015-01-01 ~ 2026-03-31
REFRESH_DATA = False
DATA_DIR = C:\pjy\SNU\2026Q1\snuquant\SNUQuant-Team3\data


## 2. 공통 함수 (resample_monthly)

In [3]:
def resample_monthly(price_series: "pd.DataFrame | pd.Series") -> "pd.Series":
    """일별 가격 시계열을 월말 기준으로 리샘플링.

    Parameters
    ----------
    price_series : 일별 가격 Series 또는 단일 열 DataFrame

    Returns
    -------
    pd.Series : 월말(해당 월의 마지막 날) 인덱스를 가진 월별 종가 시계열
    """
    s = price_series.squeeze()
    s = s.resample('ME').last().dropna()
    s.index = s.index.to_period('M').to_timestamp('M')
    return s

print('resample_monthly 정의 완료')

FM 공통 함수 정의 완료


---
## A. 한국 데이터

## A-1. KRX 유니버스

In [4]:
# ── A-1: KRX 유니버스 ─────────────────────────────────────────────────────────
# ⚠️ 생존편향 주의: 현재 시점 상장종목만 포함
_path = DATA_DIR / 'kr_uni.parquet'
if REFRESH_DATA or not _path.exists():
    def get_krx_universe() -> "pd.DataFrame":
        """KOSPI + KOSDAQ 전 상장종목을 수집해 통합 DataFrame 반환.

        Returns
        -------
        pd.DataFrame : 열={Code(6자리 str), Name, Market, yf_ticker, shares_fdr}
        """
        dfs = []
        for market in ['KOSPI', 'KOSDAQ']:
            df = fdr.StockListing(market).copy()
            if 'Symbol' in df.columns and 'Code' not in df.columns:
                df.rename(columns={'Symbol': 'Code'}, inplace=True)
            df['Market'] = market
            dfs.append(df)
        uni = pd.concat(dfs, ignore_index=True)
        uni['Code'] = uni['Code'].astype(str).str.zfill(6)
        return uni.drop_duplicates('Code').reset_index(drop=True)

    kr_uni = get_krx_universe()
    suffix_map = {'KOSPI': '.KS', 'KOSDAQ': '.KQ'}
    kr_uni['yf_ticker'] = kr_uni.apply(
        lambda r: r['Code'] + suffix_map.get(r['Market'], '.KS'), axis=1
    )
    stocks_col = next((c for c in ['Stocks', 'shares', 'ListingShares'] if c in kr_uni.columns), None)
    kr_uni['shares_fdr'] = pd.to_numeric(kr_uni[stocks_col], errors='coerce') if stocks_col else np.nan

    kr_uni.to_parquet(_path)
    print(f'저장: {_path}  shape={kr_uni.shape}')
else:
    kr_uni = pd.read_parquet(_path)
    print(f'로드: {_path}  shape={kr_uni.shape}')

print(f'KOSPI: {(kr_uni.Market=="KOSPI").sum():,}  KOSDAQ: {(kr_uni.Market=="KOSDAQ").sum():,}  합계: {len(kr_uni):,}')


로드: data\kr_uni.parquet  shape=(2773, 20)
KOSPI: 950  KOSDAQ: 1,823  합계: 2,773


## A-2. 한국 주가 (FDR)

In [ ]:
# ── A-2: 한국 주가 다운로드 (FDR, 병렬) ─────────────────────────────────────
# 월말 수정주가 행렬을 저장. 수익률 계산·클리핑은 02_analysis에서 수행.
# 순차 다운로드 대신 ThreadPoolExecutor로 병렬화 → 수십 배 빠름.
# KR_WORKERS: Naver 서버 부하 고려해 10~20 권장 (과도하면 연결 거부).
KR_WORKERS = 15

_path = DATA_DIR / 'kr_price.parquet'
if REFRESH_DATA or not _path.exists():
    kr_ticker_list = kr_uni['Code'].tolist()
    print(f'{len(kr_ticker_list):,}개 한국 종목 병렬 다운로드 (workers={KR_WORKERS})...')

    def _fetch_one(ticker: str) -> "tuple[str, pd.Series | None]":
        """단일 종목 FDR 다운로드 → 월말 수익률 Series 반환.

        Parameters
        ----------
        ticker : KRX 종목코드 (6자리 str, 예: '005930')

        Returns
        -------
        (ticker, monthly_series) 또는 (ticker, None) — 실패/데이터 부족 시
        """
        try:
            raw = fdr.DataReader(ticker, DATE_START, DATE_END)
            if raw.empty:
                return ticker, None
            col     = 'Adj Close' if 'Adj Close' in raw.columns else 'Close'
            s       = raw[col].replace(0, np.nan).dropna()
            monthly = resample_monthly(s)
            if len(monthly) >= BETA_MIN_PERIODS:
                return ticker, monthly
        except Exception:
            pass
        return ticker, None

    kr_price_dict = {}
    fail_count    = 0
    with ThreadPoolExecutor(max_workers=KR_WORKERS) as ex:
        futures = {ex.submit(_fetch_one, t): t for t in kr_ticker_list}
        for f in tqdm(as_completed(futures), total=len(futures), desc='KR Price'):
            ticker, series = f.result()
            if series is not None:
                kr_price_dict[ticker] = series
            else:
                fail_count += 1

    print(f'수집 성공: {len(kr_price_dict):,}종목  실패/제외: {fail_count}종목')
    kr_price = pd.DataFrame(kr_price_dict).sort_index()
    kr_price.to_parquet(_path)
    print(f'저장: {_path}  shape={kr_price.shape}')
else:
    kr_price = pd.read_parquet(_path)
    print(f'로드: {_path}  shape={kr_price.shape}')


2,773개 한국 종목 병렬 다운로드 (workers=15)...


KR Price:  38%|███▊      | 1062/2773 [02:35<05:09,  5.52it/s]

## A-3. KOSPI 시장수익률

In [ ]:
# ── A-3: KOSPI 시장수익률 ─────────────────────────────────────────────────────
_path = DATA_DIR / 'kr_mkt_rtn.parquet'
if REFRESH_DATA or not _path.exists():
    ks11_raw   = fdr.DataReader('KS11', DATE_START, DATE_END)['Close']
    kr_mkt_rtn = resample_monthly(ks11_raw).pct_change().dropna()
    kr_mkt_rtn.name = 'KR_market'
    kr_mkt_rtn.to_frame().to_parquet(_path)
    print(f'저장: {_path}  {len(kr_mkt_rtn)}개월')
else:
    kr_mkt_rtn = pd.read_parquet(_path)['KR_market']
    print(f'로드: {_path}  {len(kr_mkt_rtn)}개월')

print(f'KOSPI 월평균: {kr_mkt_rtn.mean()*100:.2f}%')


## A-4. FDR listing + DART corp_code 매핑

In [ ]:
# ── A-4: FDR listing + DART corp_code 매핑 ───────────────────────────────────
# kr_listing: 발행주식수(Stocks)·시가총액(Marcap) 포함한 종목 메타데이터
# corp_map   : stock_code ↔ corp_code 매핑 (DART 배치 조회에 필요)
import OpenDartReader

_path_listing  = DATA_DIR / 'kr_listing.parquet'
_path_corpmap  = DATA_DIR / 'kr_corp_map.parquet'

if REFRESH_DATA or not _path_listing.exists():
    dart = OpenDartReader(DART_API_KEY)

    kr_listing_raw = []
    for market in ['KOSPI', 'KOSDAQ']:
        df = fdr.StockListing(market).copy()
        if 'Symbol' in df.columns and 'Code' not in df.columns:
            df.rename(columns={'Symbol': 'Code'}, inplace=True)
        df['Market'] = market
        kr_listing_raw.append(df)

    kr_listing = pd.concat(kr_listing_raw, ignore_index=True)
    kr_listing['Code'] = kr_listing['Code'].astype(str).str.zfill(6)
    kr_listing = kr_listing.drop_duplicates('Code').set_index('Code')
    for col in ['Stocks', 'Marcap', 'Close']:
        if col in kr_listing.columns:
            kr_listing[col] = pd.to_numeric(kr_listing[col], errors='coerce')

    valid_codes = [t for t in kr_listing.index if t in kr_price.columns]
    kr_listing  = kr_listing.loc[valid_codes]
    kr_listing  = kr_listing[(kr_listing.get('Stocks', pd.Series([1]*len(kr_listing))) > 0)]

    corp_map_df = dart.corp_codes.copy()
    corp_map_df['stock_code'] = corp_map_df['stock_code'].astype(str).str.zfill(6)
    corp_map_df = corp_map_df[
        corp_map_df['stock_code'].str.match(r'^\d{6}$') &
        corp_map_df['stock_code'].ne('000000')
    ].drop_duplicates('stock_code')

    kr_listing.to_parquet(_path_listing)
    corp_map_df.to_parquet(_path_corpmap)
    print(f'저장: {_path_listing}  {len(kr_listing):,}종목')
    print(f'저장: {_path_corpmap}  {len(corp_map_df):,}건')
else:
    kr_listing  = pd.read_parquet(_path_listing)
    corp_map_df = pd.read_parquet(_path_corpmap)
    print(f'로드: kr_listing {len(kr_listing):,}종목  |  corp_map {len(corp_map_df):,}건')


## A-5. DART 자본총계 배치 조회

In [ ]:
# ── A-5: DART fnlttMultiAcnt 배치 조회 (자본총계) ────────────────────────────
# fnlttMultiAcnt: corp_code 최대 100개를 쉼표로 묶어 한 번에 조회.
# 연결재무제표(CFS) 우선, 미수집 종목은 별도재무제표(OFS)로 보완.
# 약 2,000종목 × 11년 → ~275회 API 호출, 예상 약 2분.
import requests

YEARS = list(range(2015, 2026))
_path = DATA_DIR / 'kr_equity.parquet'

if REFRESH_DATA or not _path.exists():
    valid_stock_codes = [t for t in kr_price.columns if t in corp_map_df['stock_code'].values]
    corp_filtered = corp_map_df[corp_map_df['stock_code'].isin(valid_stock_codes)]
    stock_to_corp = corp_filtered.set_index('stock_code')['corp_code'].to_dict()
    corp_to_stock = {v: k for k, v in stock_to_corp.items()}
    corp_codes_all = list(stock_to_corp.values())
    print(f"조회 대상 corp_code: {len(corp_codes_all):,}개")

    def fetch_equity_batch(
        corp_codes: "list[str]",
        year: int,
        fs_div: str = 'CFS',
    ) -> "pd.DataFrame":
        """fnlttMultiAcnt로 최대 100개 corp_code의 자본총계를 한 번에 조회.

        Parameters
        ----------
        corp_codes : DART corp_code 리스트 (최대 100개)
        year       : 사업연도
        fs_div     : 'CFS'=연결재무제표, 'OFS'=별도재무제표

        Returns
        -------
        pd.DataFrame : 응답 list를 DataFrame으로 변환. 실패 시 빈 DataFrame.
        """
        url    = 'https://opendart.fss.or.kr/api/fnlttMultiAcnt.json'
        params = {
            'crtfc_key':  DART_API_KEY,
            'corp_code':  ','.join(corp_codes),
            'bsns_year':  str(year),
            'reprt_code': '11011',
            'account_nm': '자본총계',
        }
        try:
            r    = requests.get(url, params=params, timeout=30)
            data = r.json()
            if data.get('status') == '000' and data.get('list'):
                df = pd.DataFrame(data['list'])
                df['fs_div'] = fs_div
                return df
        except Exception as e:
            print(f"  오류: {e}")
        return pd.DataFrame()

    def get_equity_year(year: int, batch_size: int = 100) -> "pd.Series":
        """연결재무제표(CFS) 우선, 미수집 종목은 별도재무제표(OFS)로 보완.

        Parameters
        ----------
        year       : 사업연도
        batch_size : 배치당 corp_code 수 (최대 100)

        Returns
        -------
        pd.Series : 인덱스=stock_code(6자리 str), 값=자본총계(원)
        """
        batches = [corp_codes_all[i:i+batch_size]
                   for i in range(0, len(corp_codes_all), batch_size)]
        cfs_rows = []
        for batch in batches:
            df = fetch_equity_batch(batch, year, 'CFS')
            if not df.empty:
                cfs_rows.append(df)
            time.sleep(0.3)

        cfs_df  = pd.concat(cfs_rows, ignore_index=True) if cfs_rows else pd.DataFrame()
        covered = set(cfs_df['corp_code'].unique()) if not cfs_df.empty else set()
        missing = [c for c in corp_codes_all if c not in covered]

        ofs_rows = []
        if missing:
            for batch in [missing[i:i+batch_size] for i in range(0, len(missing), batch_size)]:
                df = fetch_equity_batch(batch, year, 'OFS')
                if not df.empty:
                    ofs_rows.append(df)
                time.sleep(0.3)
        ofs_df = pd.concat(ofs_rows, ignore_index=True) if ofs_rows else pd.DataFrame()

        combined = pd.concat([cfs_df, ofs_df], ignore_index=True)
        if combined.empty:
            return pd.Series(dtype=float)

        combined = combined.sort_values('fs_div').drop_duplicates('corp_code', keep='first')
        combined['equity'] = pd.to_numeric(
            combined['thstrm_amount'].astype(str).str.replace(',', '').str.strip(),
            errors='coerce'
        )
        combined['stock_code'] = combined['corp_code'].map(corp_to_stock)
        combined = combined.dropna(subset=['stock_code', 'equity'])
        combined = combined[combined['equity'] > 0]
        combined['stock_code'] = combined['stock_code'].astype(str).str.zfill(6)
        return combined.set_index('stock_code')['equity']

    n_batches = len(corp_codes_all) // 100 + 1
    print(f"배치 {n_batches}개/년 × {len(YEARS)}년 = {n_batches*len(YEARS)}회, 약 {n_batches*len(YEARS)*0.4/60:.1f}분")

    equity_by_year = {}
    for year in tqdm(YEARS, desc='DART Equity'):
        s = get_equity_year(year)
        equity_by_year[year] = s
        print(f"  {year}년: {len(s):,}개사")

    book_df = pd.DataFrame(equity_by_year)
    book_df.columns = [int(c) for c in book_df.columns]
    book_df.to_parquet(_path)
    print(f'저장: {_path}  {book_df.notna().any(axis=1).sum():,}종목 커버')
else:
    book_df = pd.read_parquet(_path)
    book_df.columns = [int(c) for c in book_df.columns]
    print(f'로드: {_path}  {book_df.notna().any(axis=1).sum():,}종목 커버')


---
## B. 미국 데이터

## B-1. S&P 500 유니버스

In [ ]:
# ── B-1: S&P 500 유니버스 ────────────────────────────────────────────────────
# ⚠️ 생존편향 주의: 현재 편입 종목만 포함
import json as _json
_path = DATA_DIR / 'us_tickers.json'
if REFRESH_DATA or not _path.exists():
    sp500 = fdr.StockListing('S&P500')
    if 'Symbol' not in sp500.columns and 'Code' in sp500.columns:
        sp500.rename(columns={'Code': 'Symbol'}, inplace=True)
    us_tickers = sp500['Symbol'].dropna().tolist()
    us_tickers = [t.replace('.', '-') for t in us_tickers]
    us_tickers = list(dict.fromkeys(us_tickers))
    _path.write_text(_json.dumps(us_tickers), encoding='utf-8')
    print(f'저장: {_path}  {len(us_tickers)}종목')
else:
    us_tickers = _json.loads(_path.read_text(encoding='utf-8'))
    print(f'로드: {_path}  {len(us_tickers)}종목')


## B-2. 미국 주가 (yfinance)

In [ ]:
# ── B-2: 미국 주가 다운로드 (yfinance 배치) ──────────────────────────────────
# auto_adjust=True: 배당·액면분할 수정주가 자동 적용. 200개씩 청크 처리.
_path = DATA_DIR / 'us_price.parquet'
if REFRESH_DATA or not _path.exists():
    CHUNK = 200
    us_price_chunks = []
    for i in tqdm(range(0, len(us_tickers), CHUNK), desc='US Price Chunks'):
        chunk = us_tickers[i:i+CHUNK]
        try:
            raw = yf.download(chunk, start=DATE_START, end=DATE_END,
                              auto_adjust=True, progress=False, threads=True)
            if isinstance(raw.columns, pd.MultiIndex):
                raw = raw['Close']
            else:
                raw = raw[['Close']] if 'Close' in raw.columns else raw
            us_price_chunks.append(raw)
        except Exception as e:
            print(f'  chunk {i} 실패: {e}')
        time.sleep(0.5)

    us_price_raw = pd.concat(us_price_chunks, axis=1)
    us_price_raw = us_price_raw.loc[:, ~us_price_raw.columns.duplicated()]

    # 월말 리샘플링
    us_price = us_price_raw.resample('ME').last()
    us_price.index = us_price.index.to_period('M').to_timestamp('M')
    us_price = us_price.dropna(how='all')
    us_price.to_parquet(_path)
    print(f'저장: {_path}  shape={us_price.shape}')
else:
    us_price = pd.read_parquet(_path)
    print(f'로드: {_path}  shape={us_price.shape}')


## B-3. S&P 500 시장수익률

In [ ]:
# ── B-3: S&P 500 시장수익률 ──────────────────────────────────────────────────
_path = DATA_DIR / 'us_mkt_rtn.parquet'
if REFRESH_DATA or not _path.exists():
    sp500_raw  = yf.download('^GSPC', start=DATE_START, end=DATE_END,
                             auto_adjust=True, progress=False)['Close'].squeeze()
    us_mkt_rtn = sp500_raw.resample('ME').last().pct_change().dropna().squeeze()
    us_mkt_rtn.index = us_mkt_rtn.index.to_period('M').to_timestamp('M')
    us_mkt_rtn.name  = 'US_market'
    us_mkt_rtn.to_frame().to_parquet(_path)
    print(f'저장: {_path}  {len(us_mkt_rtn)}개월')
else:
    us_mkt_rtn = pd.read_parquet(_path)['US_market']
    print(f'로드: {_path}  {len(us_mkt_rtn)}개월')

print(f'S&P500 월평균: {float(us_mkt_rtn.mean())*100:.2f}%')


## B-4. 미국 기본데이터 (주식수 + 연도별 장부가치)

In [ ]:
# ── B-4: 미국 기본데이터 (주식수 + 연도별 장부가치) ──────────────────────────
# yfinance balance_sheet에서 최근 4개 회계연도의 Stockholders Equity 수집.
# ⚠️ 한계: 최근 4년치만 제공 → 2015~2018년 B/M 결측 가능.
import json as _json

_path_fund = DATA_DIR / 'us_fund.parquet'
_path_bv   = DATA_DIR / 'us_bv_annual.json'

if REFRESH_DATA or not _path_fund.exists():
    def fetch_us_fund_safe(ticker: str, retries: int = 3) -> "dict[str, object]":
        """단일 종목의 주식수와 연도별 Stockholders Equity를 수집.

        balance_sheet에서 우선순위 순으로 자기자본 항목을 탐색:
          1) Stockholders Equity  2) Total Stockholder Equity
          3) Common Stock Equity  4) Total Equity Gross Minority Interest
        네트워크 오류 시 지수 백오프(1→2→4초)로 최대 retries회 재시도.

        Parameters
        ----------
        ticker  : yfinance 티커 심볼
        retries : 최대 재시도 횟수

        Returns
        -------
        dict : {'ticker': str, 'shares': float | nan,
                'bv_annual': dict[int, float]}
               재시도 소진 시 shares=NaN, bv_annual={} 반환.
        """
        for attempt in range(retries):
            try:
                t_obj  = yf.Ticker(ticker)
                info   = t_obj.info
                shares = info.get('sharesOutstanding') or info.get('impliedSharesOutstanding', np.nan)
                bv_annual = {}
                bs = t_obj.balance_sheet
                if bs is not None and not bs.empty:
                    for key in ['Stockholders Equity', 'Total Stockholder Equity',
                                'Common Stock Equity', 'Total Equity Gross Minority Interest']:
                        if key in bs.index:
                            for col_date in bs.columns:
                                val = bs.loc[key, col_date]
                                if pd.notna(val) and float(val) > 0:
                                    bv_annual[col_date.year] = float(val)
                            break
                return {'ticker': ticker, 'shares': shares, 'bv_annual': bv_annual}
            except Exception:
                time.sleep(2 ** attempt)
        return {'ticker': ticker, 'shares': np.nan, 'bv_annual': {}}

    us_tickers_to_fetch = [t for t in us_tickers if t in us_price.columns]
    print(f"{len(us_tickers_to_fetch)}개 미국 종목 기본데이터 수집 (workers=3)...")

    results = []
    with ThreadPoolExecutor(max_workers=3) as ex:
        futures = {ex.submit(fetch_us_fund_safe, t): t for t in us_tickers_to_fetch}
        for f in tqdm(as_completed(futures), total=len(futures), desc='US Fundamentals'):
            results.append(f.result())
            time.sleep(0.05)

    us_fund_shares = {}
    us_bv_annual   = {}
    for r in results:
        t = r['ticker']
        if pd.notna(r['shares']) and r['shares'] > 0:
            us_fund_shares[t] = float(r['shares'])
        if r['bv_annual']:
            us_bv_annual[t] = {int(k): v for k, v in r['bv_annual'].items()}

    shares_series = pd.Series(us_fund_shares)
    valid_us = shares_series[shares_series > 0].to_frame('shares')
    valid_us = valid_us[valid_us.index.isin(us_bv_annual)]

    valid_us.to_parquet(_path_fund)
    _path_bv.write_text(_json.dumps(us_bv_annual), encoding='utf-8')
    print(f'저장: {_path_fund}  {len(valid_us)}종목')
    print(f'저장: {_path_bv}')
else:
    valid_us     = pd.read_parquet(_path_fund)
    us_bv_annual = {k: {int(yr): v for yr, v in d.items()}
                    for k, d in _json.loads(_path_bv.read_text(encoding='utf-8')).items()}
    print(f'로드: {len(valid_us)}종목 (주식수 + 연도별 장부가치)')

print(f'유효 종목: {len(valid_us)}개  (shares + bv_annual 모두 있음)')
valid_us.head(3)
